In [1]:
import mdtraj as md

from src.utils.utils import prime_factorization



In [2]:
DDP8_sample_pbd_file = "/mnt/x/School/trajects/pre_processed_data/sep_prot_frames_DPP8_apo/frame_0.pdb"
DDP9_sample_pbd_file = "/mnt/x/School/trajects/pre_processed_data/sep_prot_frames_DPP9_apo/frame_0.pdb"

DDP8_sample_pbd = md.load(DDP8_sample_pbd_file)
DDP9_sample_pbd = md.load(DDP9_sample_pbd_file)


In [3]:
DDP8_sample_coordinates = DDP8_sample_pbd.xyz
DDP9_sample_coordinates = DDP9_sample_pbd.xyz

In [4]:
DDP8_sample_coordinates

array([[[ 4.954,  6.34 , 11.709],
        [ 5.   ,  6.288, 11.782],
        [ 4.932,  6.272, 11.637],
        ...,
        [ 4.94 ,  8.415,  9.565],
        [ 4.87 ,  8.44 ,  9.663],
        [ 5.061,  8.457,  9.556]]], dtype=float32)

In [5]:
print(DDP8_sample_coordinates.shape)


(1, 27286, 3)


In [6]:
nr_atoms = DDP8_sample_coordinates.shape[1]
prime_factors = prime_factorization(nr_atoms)
print(prime_factors)

[2, 7, 1949]


In [7]:
print(nr_atoms)
start = 150
while start*start < nr_atoms:
    start += 1
print(f"{start}x{start}")
total_pixels = start*start
print(f"total pixels: {total_pixels}")
print(f"total empty pixels: {total_pixels- nr_atoms}")


27286
166x166
total pixels: 27556
total empty pixels: 270


In [8]:
print(DDP9_sample_coordinates.shape)
nr_atoms = DDP9_sample_coordinates.shape[1]
print(nr_atoms)
print(prime_factorization(nr_atoms))


(1, 26876, 3)
26876
[2, 2, 6719]


In [9]:
print(f"black pixels needed to fill image:")
print(f"\tDDP8: {total_pixels - DDP8_sample_coordinates.shape[1]}")
print(f"\tDDP9: {total_pixels - DDP9_sample_coordinates.shape[1]}")


black pixels needed to fill image:
	DDP8: 270
	DDP9: 680


In [8]:
# finding the largest dimension of the molecule
x = DDP8_sample_coordinates[:, :, 0]
y = DDP8_sample_coordinates[:, :, 1]
z = DDP8_sample_coordinates[:, :, 2]
length_x = x.max() - x.min()
length_y = y.max() - y.min()
length_z = z.max() - z.min()
print("Dimension sizef for DDP8:")
print(length_x, length_y, length_z)
largest_dimension = max(length_x, length_y, length_z)
print("Largest dimension:", largest_dimension)

Dimension sizef for DDP8:
12.327 8.354 11.801
Largest dimension: 12.327


In [9]:
x = DDP9_sample_coordinates[:, :, 0]
y = DDP9_sample_coordinates[:, :, 1]
z = DDP9_sample_coordinates[:, :, 2]
length_x = x.max() - x.min()
length_y = y.max() - y.min()
length_z = z.max() - z.min()
print("Dimension sizes for DDP9:")
print(length_x, length_y, length_z)
largest_dimension = max(length_x, length_y, length_z)
print("Largest dimension:", largest_dimension)

Dimension sizes for DDP9:
12.162 9.805 11.518001
Largest dimension: 12.162


In [12]:
import numpy as np
from src.utils.randomGenerator import RandomGenerator


randomGenerator = RandomGenerator()
radius = largest_dimension / 2
random_point = randomGenerator.random_uniform(-radius, radius, size=3)



In [11]:
scrambler = PositionScrambler()
scrambler.scramble(DDP8_sample_coordinates)

TypeError: PositionScrambler.scramble() missing 1 required positional argument: 'diameter'

In [3]:
import torchvision
from src.utils.configParser import ConfigParser
import os
import torch
from torchvision import transforms
def analyze_dataset():
    """Comprehensive dataset analysis"""
    print("=== DATASET ANALYSIS ===")
    config_parser  = ConfigParser("config.ini")
    dataset_folder = config_parser.get("Model Training", "Input Folder")
    train_folder = os.path.join(dataset_folder, 'train')
    test_folder = os.path.join(dataset_folder, 'test')
    validate_folder = os.path.join(dataset_folder, 'val')
    transform = transforms.Compose([
        transforms.ToTensor(),
    ])
    trainset = torchvision.datasets.ImageFolder(root=train_folder, transform=transform)
    testset = torchvision.datasets.ImageFolder(root=test_folder, transform=transform)

    batch_size = 16
    trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size, shuffle=True, num_workers=8)
    testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size, shuffle=False, num_workers=8)

    # Check train set
    class_counts = [0] * 5
    for images, labels in trainloader:
        for label in labels:
            class_counts[label.item()] += 1

    print(f"Train set class distribution: {class_counts}")
    print(f"Total training samples: {sum(class_counts)}")

    # Check test set
    test_counts = [0] * 5
    for images, labels in testloader:
        for label in labels:
            test_counts[label.item()] += 1
    print(f"Test set class distribution: {test_counts}")
    print(f"Total test samples: {sum(test_counts)}")

    # Check image statistics
    for images, _ in trainloader:
        print(f"Image shape: {images.shape}")
        print(f"Image range: [{images.min():.3f}, {images.max():.3f}]")
        print(f"Image mean: {images.mean():.3f}, std: {images.std():.3f}")
        break
analyze_dataset()

=== DATASET ANALYSIS ===
Train set class distribution: [135, 146, 144, 139, 144]
Total training samples: 708
Test set class distribution: [32, 28, 35, 26, 30]
Total test samples: 151
Image shape: torch.Size([16, 3, 166, 166])
Image range: [0.000, 1.000]
Image mean: 0.487, std: 0.220
